In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv('cause_death.csv')
df.head()

,Cause,Type,Year,Data Redundancy,Total Deaths,Source,Page at Source,Source URL
0,Abrasi Air Laut,Bencana Alam,2012,1,0,Profil Kesehatan Indonesia Tahun 2012,Lampiran 3.38,https://pusdatin.kemkes.go.id/download.php?fil...
1,AIDS,Bencana Non Alam dan Penyakit,2000,1,47,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
2,AIDS,Bencana Non Alam dan Penyakit,2001,1,99,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
3,AIDS,Bencana Non Alam dan Penyakit,2002,1,100,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
4,AIDS,Bencana Non Alam dan Penyakit,2003,1,261,Profil Kesehatan Indonesia Tahun 2010,115,https://pusdatin.kemkes.go.id/download.php?fil...


In [25]:
df.fillna(0)
df.drop_duplicates()

q1 = df["Total Deaths"].quantile(0.25)
q3 = df["Total Deaths"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr 
upper = q3 + 1.5 * iqr 

df_clean = df[(df["Total Deaths"] >= lower) & (df["Total Deaths"] <= upper)]

print(f"Data sebelum cleaning : {len(df)}")
print(f"Data sesudah cleaning : {len(df_clean)}")

Data sebelum cleaning : 680
Data sesudah cleaning : 552


In [33]:
from scipy import stats

# 1. Uji Shapiro-Wilk untuk normalitas

shapiro_test = stats.shapiro(df_clean['Total Deaths'])
print("Hasil Uji Shapiro-Wilk")
print("Statistik : ", shapiro_test.statistic)

Hasil Uji Shapiro-Wilk
Statistik :  0.5902771802575988


In [27]:
# 2. Uji Kruskal-Wallis

kruskal_result = stats.kruskal(df_clean[df_clean['Type'] == 'Bencana Alam']['Total Deaths'],
                                 df_clean[df_clean['Type'] == 'Bencana Non Alam dan Penyakit']['Total Deaths'])

print("Hasil Uji Kruskal-Wallis:", kruskal_result)


Hasil Uji Kruskal-Wallis: KruskalResult(statistic=12.181545514918252, pvalue=0.0004826463828865544)


In [28]:
# 3. Korelasi Pearson

from scipy import stats

correlation, p_value = stats.pearsonr(df_clean["Year"], df_clean["Total Deaths"])

print(f"Korelasi antara Year dan Total Deaths: {correlation}")
print(f"P-value: {p_value}")


Korelasi antara Year dan Total Deaths: -0.10708174138273542
P-value: 0.011822788193798232


In [29]:
import statsmodels.api as sm

# Memisahkan variabel independen (Year) dan dependen (Total Deaths)
X = df_clean['Year']  # variabel independen
y = df_clean['Total Deaths']  # variabel dependen

# Menambahkan konstanta untuk menghitung intercept
X = sm.add_constant(X)

# Membuat model regresi
model = sm.OLS(y, X).fit()

# Menampilkan ringkasan hasil regresi
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           Total Deaths   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     6.380
Date:                Mon, 28 Oct 2024   Prob (F-statistic):             0.0118
Time:                        15:29:47   Log-Likelihood:                -3524.2
No. Observations:                 552   AIC:                             7052.
Df Residuals:                     550   BIC:                             7061.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5362.2140   2091.692      2.564      0.0

In [30]:
import pingouin as pg
anova_result = pg.anova(dv='Total Deaths', between='Type', data=df)
print(anova_result)

  Source  ddof1  ddof2         F     p-unc       np2
0   Type      2    677  0.428558  0.651624  0.001264
